In [ ]:
# Test for Assignment 2
import Pkg
c = ["Metal", "Chain", "BenchmarkTools"]
Pkg.add(c)
using Metal, Chain, BenchmarkTools

# カーネル関数を使った3次元配列代入 #2_2
function kernel_assign_2_2(ac_x::MtlDeviceArray{Float32, 3}, ac_y::MtlDeviceArray{Float32, 3},
                            xx::MtlDeviceArray{Float32, 1}, yy::MtlDeviceArray{Float32, 1},
                            x0::MtlDeviceArray{Float32, 1}, y0::MtlDeviceArray{Float32, 1})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i <= size(ac_x, 1) && j <= size(ac_x, 2) && k <= size(ac_x, 3)
        ac_x[i, j, k] = x0[k] - xx[i]
        ac_y[i, j, k] = y0[k] - yy[i]

        return nothing

    end

    return nothing

end

# カーネル関数を使った3次元配列代入 #2_3
function kernel_assign_2_3(cd_x::MtlDeviceArray{Float32, 3}, cd_y::MtlDeviceArray{Float32, 3},
                            x0::MtlDeviceArray{Float32, 1}, y0::MtlDeviceArray{Float32, 1},
                            x1::MtlDeviceArray{Float32, 1}, y1::MtlDeviceArray{Float32, 1})::Nothing

    i = thread_position_in_grid_3d().x
    j = thread_position_in_grid_3d().y
    k = thread_position_in_grid_3d().z

    if i <= size(cd_x, 1) && j <= size(cd_x, 2) && k <= size(cd_x, 3)
        cd_x[i, j, k] = x1[k] - x0[k]
        cd_y[i, j, k] = y1[k] - y0[k]

        return nothing

    end

    return nothing

end


num_points = 31209
num_edges = 176
    # num_points = 312
    # num_edges = 176
    
α = 0.5
div_n = @chain Metal.current_device() begin
    _.maxBufferLength
    floor(α * (_ / (sizeof(Float32) * num_points * num_edges)))
    convert(Int, _)
end

Mtl_xx = MtlArray{Float32}(rand(Float32, num_points))
Mtl_yy = MtlArray{Float32}(rand(Float32, num_points))
Mtl_x0 = MtlArray{Float32}(rand(Float32, num_edges))
Mtl_y0 = MtlArray{Float32}(rand(Float32, num_edges))
Mtl_x1 = MtlArray{Float32}(rand(Float32, num_edges))
Mtl_y1 = MtlArray{Float32}(rand(Float32, num_edges))
    
# [点, 角度, 辺]
Mtl_ac_x = MtlArray{Float32}(undef, num_points, div_n, num_edges)
Mtl_ac_y = similar(Mtl_ac_x)
Mtl_cd_x = similar(Mtl_ac_x)
Mtl_cd_y = similar(Mtl_ac_x)
Mtl_ca_x = similar(Mtl_ac_x)
Mtl_ca_y = similar(Mtl_ac_x)
    
# Threadgroup size configuration
# threads_per_group = (10, 10, 10)  # MacBook Pro M4Max OK, Mac Studio NG
threads_per_group = (4, 4, 1)  #  <= 640
# threads_per_group = (8, 8, 4)  #  = 2^n MacStudio 終了後にパソコンが再起動される．
# threads_per_group = (8, 4, 4)  #  = 2^n MacBook M4Max NG
# threads_per_group = (8, 8, 4)  #  = 2^n MacStudio 終了後にパソコンが再起動される．
num_groups = (ceil(Int, num_points / threads_per_group[1]), 
            ceil(Int, div_n / threads_per_group[2]), 
            ceil(Int, num_edges / threads_per_group[3]))
# Kernel function execution
@metal threads = threads_per_group groups = num_groups kernel_assign_2_2(Mtl_ac_x, Mtl_ac_y, 
                                                                        Mtl_xx, Mtl_yy,
                                                                        Mtl_x0, Mtl_y0)
    
@metal threads = threads_per_group groups = num_groups kernel_assign_2_3(Mtl_cd_x, Mtl_cd_y, 
                                                                        Mtl_x0, Mtl_y0,
                                                                        Mtl_x1, Mtl_y1)

Mtl_ca_x .= -Mtl_ac_x
Mtl_ca_y .= -Mtl_ac_y


# xx = Array(Mtl_xx)
# yy = Array(Mtl_yy)
# x0 = Array(Mtl_x0)
# y0 = Array(Mtl_y0)
# x1 = Array(Mtl_x1)
# y1 = Array(Mtl_y1)
# ac_x = Array(Mtl_ac_x)
# ac_y = Array(Mtl_ac_y)
# cd_x = Array(Mtl_cd_x)
# cd_y = Array(Mtl_cd_y)
ca_x = Array(Mtl_ca_x)
ca_y = Array(Mtl_ca_y)
# # Check the results
# @show (xx[1:3], yy[1:3])
# @show (x0[10], y0[10])
# @show (x1[10], y1[10])
# @show (ac_x[1:3, 1, 10], ac_y[1:3, 1, 10])
@show (ca_x[1:3, 1, 10], ca_y[1:3, 1, 10])
# @show (cd_x[1:3, 1, 10], cd_y[1:3, 1, 10])
# @show (xx[1:3] - x0[10], yy[1:3] - y0[10])
# @show (x1[10] - x0[10], y1[10] - y0[10])

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
